# Tarea 2
- Autor: Mauricio Montes
- Fecha: 26 de junio de 2025

## Imports
Para la tarea son necesarios las librerías `pandas` y `plotly`.

In [1]:
# %pip install pandas plotly
import pandas as pd
import plotly.express as plt

## Carga de datos
Tras descargar el dataset del covid 19, hay que cargarlo en un `DataFrame`.

In [2]:
df = pd.read_csv("time_series_covid19_confirmed_global.csv")
df.head()

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,NaN,Afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,NaN,Albania,41.15330,20.168300,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,NaN,Algeria,28.03390,1.659600,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,NaN,Andorra,42.50630,1.521800,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,NaN,Angola,-11.20270,17.873900,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288


# Transformación de datos
Primero, realizamos un análisis exploratorio del dataset:

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 289 entries, 0 to 288
Columns: 1147 entries, Province/State to 3/9/23
dtypes: float64(2), int64(1143), object(2)
memory usage: 2.5+ MB


In [4]:
df['Province/State'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 289 entries, 0 to 288
Series name: Province/State
Non-Null Count  Dtype 
--------------  ----- 
91 non-null     object
dtypes: object(1)
memory usage: 2.4+ KB


El dataframe consiste en una tabla de 1147 columnas, en donde buena parte de las columnas son fechas (algo que se puede ver usando el método `head`). Se entiende que son fechas, y que son los casos confirmados de COVID-19 por cada fecha... Pero `pandas` los considera datos de tipo `objeto`, no `datetime`. Hay que realizar esa transformación.

In [5]:
# Las columnas de fecha empiezan después de la cuarta columna
date_columns = df.columns[4:]
df.rename(columns={col: pd.to_datetime(col, format="%m/%d/%y") for col in date_columns}, inplace=True)
df

,Province/State,Country/Region,Lat,Long,2020-01-22 00:00:00,2020-01-23 00:00:00,2020-01-24 00:00:00,2020-01-25 00:00:00,2020-01-26 00:00:00,2020-01-27 00:00:00,...,2023-02-28 00:00:00,2023-03-01 00:00:00,2023-03-02 00:00:00,2023-03-03 00:00:00,2023-03-04 00:00:00,2023-03-05 00:00:00,2023-03-06 00:00:00,2023-03-07 00:00:00,2023-03-08 00:00:00,2023-03-09 00:00:00
0,NaN,Afghanistan,33.939110,67.709953,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,NaN,Albania,41.153300,20.168300,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,NaN,Algeria,28.033900,1.659600,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,NaN,Andorra,42.506300,1.521800,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,NaN,Angola,-11.202700,17.873900,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284,NaN,West Bank and Gaza,31.952200,35.233200,0,0,0,0,0,0,...,703228,703228,703228,703228,703228,703228,703228,703228,703228,703228
285,NaN,Winter Olympics 2022,39.904200,116.407400,0,0,0,0,0,0,...,535,535,535,535,535,535,535,535,535,535
286,NaN,Yemen,15.552727,48.516388,0,0,0,0,0,0,...,11945,11945,11945,11945,11945,11945,11945,11945,11945,11945
287,NaN,Zambia,-13.133897,27.849332,0,0,0,0,0,0,...,343012,343012,343079,343079,343079,343135,343135,343135,343135,343135


Para poder analizarlo mejor, se me pide que agrupe los datos en meses... Pero la función `groupby` solo funciona con una columna clave; así que no es algo que puedo hacer directamente. Lo que puedo hacer es usar la función `melt`, que básicamente tendrá un montón de filas... Una fila por cada fecha.

In [6]:
df_melted = df.melt(
    id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'], 
    var_name='Date',
    value_name='Confirmed'
)

# El dataframe ahora tiene 6 columnas, y 330327 filas
df_melted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330327 entries, 0 to 330326
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Province/State  104013 non-null  object 
 1   Country/Region  330327 non-null  object 
 2   Lat             328041 non-null  float64
 3   Long            328041 non-null  float64
 4   Date            330327 non-null  object 
 5   Confirmed       330327 non-null  int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 15.1+ MB


In [7]:
df_melted['Date'] = pd.to_datetime(df_melted['Date'])

df_melted[
    (df_melted['Country/Region'] == 'El Salvador') &
    (df_melted['Date'].dt.year == 2020) &
    (df_melted['Date'].dt.month == 3)
]

,Province/State,Country/Region,Lat,Long,Date,Confirmed
11383,NaN,El Salvador,13.7942,-88.8965,2020-03-01,0
11672,NaN,El Salvador,13.7942,-88.8965,2020-03-02,0
11961,NaN,El Salvador,13.7942,-88.8965,2020-03-03,0
12250,NaN,El Salvador,13.7942,-88.8965,2020-03-04,0
12539,NaN,El Salvador,13.7942,-88.8965,2020-03-05,0
12828,NaN,El Salvador,13.7942,-88.8965,2020-03-06,0
13117,NaN,El Salvador,13.7942,-88.8965,2020-03-07,0
13406,NaN,El Salvador,13.7942,-88.8965,2020-03-08,0
13695,NaN,El Salvador,13.7942,-88.8965,2020-03-09,0
13984,NaN,El Salvador,13.7942,-88.8965,2020-03-10,0


In [8]:
# Contando los valores únicos para 'Country/Region' en df_melted son 201, que eran la cantidad de países que el dataframe original tiene
print(df_melted['Country/Region'].nunique())
print(df['Country/Region'].nunique() == df_melted['Country/Region'].nunique())

201
True


In [9]:
# Mientras tanto, la columna 'Date' tiene 1143 valores únicos... Que son las 1147 - 4 columnas que tenía el dataframe inicialmente.
print(df_melted['Date'].nunique())

1143


Bueno, ahora que sé cómo funciona la función `melt`, continuamos con la agrupación por meses...

In [10]:
df = df.melt(id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'], var_name='Date', value_name='Confirmed')

In [11]:
df['Date'] = pd.to_datetime(df['Date'])
df['YearMonth'] = df['Date'].dt.to_period('M')
result = df.groupby(['Country/Region', 'YearMonth'])['Confirmed'].sum().reset_index()
result

,Country/Region,YearMonth,Confirmed
0,Afghanistan,2020-01,0
1,Afghanistan,2020-02,30
2,Afghanistan,2020-03,1141
3,Afghanistan,2020-04,25152
4,Afghanistan,2020-05,222720
...,...,...,...
7834,Zimbabwe,2022-11,7738061
7835,Zimbabwe,2022-12,8046576
7836,Zimbabwe,2023-01,8072411
7837,Zimbabwe,2023-02,7370790


## Visualización de datos

Ya que el dataset está organizado en países y meses del año, ahora selecciono tres países:
- Estados Unidos (US)
- Reino Unido (United Kindgom)
- El Salvador

In [12]:
countries = ['US', 'United Kingdom', 'El Salvador']
subset = result[result['Country/Region'].isin(countries)]
subset

,Country/Region,YearMonth,Confirmed
2145,El Salvador,2020-01,0
2146,El Salvador,2020-02,0
2147,El Salvador,2020-03,156
2148,El Salvador,2020-04,5333
2149,El Salvador,2020-05,41262
...,...,...,...
7444,United Kingdom,2022-11,725591181
7445,United Kingdom,2022-12,753615408
7446,United Kingdom,2023-01,758377658
7447,United Kingdom,2023-02,687682836


Creamos el gráfico de líneas interactivo:

In [13]:
subset['YearMonth'] = subset['YearMonth'].dt.to_timestamp()

fig = plt.line(
    subset,
    x = 'YearMonth',
    y = 'Confirmed',
    color = 'Country/Region',
    title = 'Casos confirmados por mes'
)

fig.update_layout(
    xaxis_title = 'Fecha',
    yaxis_title = 'Casos confirmados',
    hovermode = 'x unified'
)

fig.show()

C:\Users\MAURICIO MONTES\AppData\Local\Temp\ipykernel_21620\1278237848.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset['YearMonth'] = subset['YearMonth'].dt.to_timestamp()


Ok, eh, hay un problema con los datos. La idea es que la gráfica muestre la cantidad de casos confirmados por país, y para eso se usó la función `groupby` por país y por mes; para así hacer la línea de tiempo. Lo que se hizo con los datos fue sumarlos. Ahora... la situación es que... Estados Unidos tiene 3.15 mil millones de casos confirmados para Enero del 2023. Más casos que población.
Esto significa que los datos obtenidos no son de los casos confirmados nuevos por día... Sino que es una acumulación de los casos confirmados por cada día. Ya se hace una suma de los casos de días anteriores para un nuevo día; o esa es una posibilidad. Entonces, puede que lo mejor sea obtener el máximo por cada mes.

In [14]:
real_result = df.groupby(['Country/Region', 'YearMonth'])['Confirmed'].max().reset_index()

new_subset = real_result[real_result['Country/Region'].isin(countries)]
new_subset['YearMonth'] = new_subset['YearMonth'].dt.strftime('%Y-%m')

fig = plt.line(
    new_subset,
    x = 'YearMonth',
    y = 'Confirmed',
    color = 'Country/Region',
    title = 'Casos confirmados por mes'
)

fig.update_layout(
    xaxis_title = 'Fecha',
    yaxis_title = 'Casos confirmados',
    hovermode = 'x unified'
)

fig.show()

C:\Users\MAURICIO MONTES\AppData\Local\Temp\ipykernel_21620\805162835.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Nuevo problema, ahora no se observa un pico. Esto confirma que los datos son acumulativos, registrando la suma de los casos confirmados. Entonces, para obtener la cantidad de casos confirmados por día, tendría que hacerse una diferencia entre los días: La cantidad de casos confirmados de este día menos los de ayer; o en este caso, los meses.

In [ ]:
for_real_result = df.groupby(['Country/Region', 'YearMonth'])['Confirmed'].max().reset_index()
for_real_result['NewCases'] = for_real_result.groupby(['Country/Region'])['Confirmed'].diff().fillna(0)

real_subset = for_real_result[for_real_result['Country/Region'].isin(countries)]
real_subset['YearMonth'] = real_subset['YearMonth'].dt.strftime('%Y-%m')

fig = plt.line(
    real_subset,
    x = 'YearMonth',
    y = 'NewCases',
    color = 'Country/Region',
    title = 'Nuevos casos confirmados por mes'
)

fig.update_layout(
    xaxis_title = 'Fecha',
    yaxis_title = 'Casos mensuales',
    hovermode = 'x unified'
)

fig.show()

C:\Users\MAURICIO MONTES\AppData\Local\Temp\ipykernel_21620\3814959157.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
el_salvador_subset = for_real_result[for_real_result['Country/Region'] == 'El Salvador']
el_salvador_subset['YearMonth'] = el_sa['YearMonth'].dt.strftime('%Y-%m')

fig = plt.line(
    el_salvador_subset,
    x = 'YearMonth',
    y = 'NewCases',
    title = 'Nuevos casos confirmados por mes'
)

fig.update_layout(
    xaxis_title = 'Fecha',
    yaxis_title = 'Casos mensuales',
    hovermode = 'x unified'
)

fig.show()

C:\Users\MAURICIO MONTES\AppData\Local\Temp\ipykernel_21620\1447917562.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

